# 00. Download the Replogle Subset and Checkpoint

This notebook prepares the minimal Replogle assets needed to run the PerturbDiff reproduction workflow.

The key idea is to start with a compact perturbation dataset before scaling to larger settings. The download includes the processed Replogle `.h5ad`, the selected 2,000 HVG gene list, gene embeddings, index caches, metadata, and the official Replogle checkpoint.

In [1]:
from pathlib import Path

# Set this to the PerturbDiff repository root.
# If the notebook server is launched from the repo root, the default is already correct.
PROJECT_ROOT = Path("/home/lwf/PerturbDiff").resolve()
assert (PROJECT_ROOT / "src" / "apps" / "run" / "rawdata_diffusion_sampling.py").exists(), \
    "Set PROJECT_ROOT to the PerturbDiff repository root before continuing."

DATASET_NAME = "replogle"
DATA_ROOT = PROJECT_ROOT / "data" / DATASET_NAME
PERTURB_ROOT = DATA_ROOT / "perturb_data"
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / DATASET_NAME

REPO_DATA = "katarinayuan/PerturbDiff_data"
REPO_CKPT = "katarinayuan/PerturbDiff_release_ckpt"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
PERTURB_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_NAME:", DATASET_NAME)
print("PERTURB_ROOT:", PERTURB_ROOT)
print("CKPT_ROOT:", CKPT_ROOT)

PROJECT_ROOT: /home/lwf/PerturbDiff
DATASET_NAME: replogle
PERTURB_ROOT: /home/lwf/PerturbDiff/data/replogle/perturb_data
CKPT_ROOT: /home/lwf/PerturbDiff/checkpoints/replogle


## 1. Configure HuggingFace Download Settings

SSL EOF errors usually come from an interrupted HTTPS connection. If direct access to HuggingFace is unstable, use a mirror endpoint or configure a proxy before importing `huggingface_hub`.

In [2]:
import os

# Use a mirror when direct HuggingFace access is unstable.
# Set HF_ENDPOINT = None to use the official endpoint: https://huggingface.co
HF_ENDPOINT = "https://hf-mirror.com"

# Optional proxy example:
# PROXIES = {"http": "http://127.0.0.1:7890", "https": "http://127.0.0.1:7890"}
PROXIES = None

if HF_ENDPOINT is not None:
    os.environ["HF_ENDPOINT"] = HF_ENDPOINT
else:
    os.environ.pop("HF_ENDPOINT", None)

print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT", "https://huggingface.co"))
print("PROXIES:", PROXIES)

HF_ENDPOINT: https://hf-mirror.com
PROXIES: None


## 2. Download the Minimal Replogle Assets

`snapshot_download` is configured with `allow_patterns` so that only the Replogle-related files and shared metadata are requested.

In [ ]:
# If huggingface_hub is not installed, run:
# !pip install -U "huggingface_hub[cli]" hf_xet

import time
from huggingface_hub import snapshot_download

def snapshot_download_with_retry(*, max_attempts=5, sleep_seconds=20, **kwargs):
    for attempt in range(1, max_attempts + 1):
        try:
            return snapshot_download(
                **kwargs,
                endpoint=HF_ENDPOINT,
                proxies=PROXIES,
                etag_timeout=60,
                max_workers=1,
            )
        except Exception as exc:
            if attempt == max_attempts:
                raise
            print(f"Download failed on attempt {attempt}/{max_attempts}: {type(exc).__name__}: {exc}")
            print(f"Retrying in {sleep_seconds} seconds...")
            time.sleep(sleep_seconds)

data_patterns = [
    "perturb_data/finetune_data/nadig_processed_data/**",
    "perturb_data/selected_genes/**",
    "perturb_data/gene_names/**",
    "perturb_data/indices_cache/**",
    "perturb_data/meta_data/**",
]

snapshot_download_with_retry(
    repo_id=REPO_DATA,
    repo_type="dataset",
    local_dir=DATA_ROOT,
    allow_patterns=data_patterns,
)


print("Replogle-related data downloaded to:", DATA_ROOT)

## 3. Decompress the `.zst` File if Needed

The processed dataset may appear either as `replogle.h5ad` or `replogle.h5ad.zst`. The downstream notebooks expect the `.h5ad` file.

In [4]:
import subprocess

replogle_h5ad = PERTURB_ROOT / "finetune_data" / "nadig_processed_data" / "replogle.h5ad"
replogle_zst = replogle_h5ad.with_suffix(replogle_h5ad.suffix + ".zst")

if replogle_h5ad.exists():
    print("Found:", replogle_h5ad)
elif replogle_zst.exists():
    print("Found compressed file:", replogle_zst)
    print("Decompressing with zstd. If this fails, install zstd first: mamba install -c conda-forge zstd")
    subprocess.run(["zstd", "-d", str(replogle_zst)], check=True)
else:
    raise FileNotFoundError(f"Cannot find {replogle_h5ad} or {replogle_zst}")

print("Ready:", replogle_h5ad)

Found: /home/lwf/PerturbDiff/data/replogle/perturb_data/finetune_data/nadig_processed_data/replogle.h5ad
Ready: /home/lwf/PerturbDiff/data/replogle/perturb_data/finetune_data/nadig_processed_data/replogle.h5ad


## 4. Download the Replogle Checkpoint

The official `from_scratch_replogle.ckpt` checkpoint matches the Replogle 2,000 HVG setup and is a convenient starting point for inference.

In [ ]:
snapshot_download_with_retry(
    repo_id=REPO_CKPT,
    repo_type="model",
    local_dir=CKPT_ROOT,
    allow_patterns=["from_scratch_replogle.ckpt"],
)

ckpt_path = CKPT_ROOT / "from_scratch_replogle.ckpt"
print("Checkpoint:", ckpt_path)
print("Exists:", ckpt_path.exists())

## 5. Verify the Downloaded Files

The following check lists the files required by the remaining notebooks.

In [5]:
required_paths = {
    "Replogle h5ad": replogle_h5ad,
    "selected genes": PERTURB_ROOT / "selected_genes" / "replogle_real_selected_genes.pkl",
    "Replogle perturbation embedding": PERTURB_ROOT / "gene_names" / "replogle_gene_emb_dict_perturbation_emb_dict.pkl",
    # "checkpoint": ckpt_path,
}

for name, path in required_paths.items():
    status = "OK" if path.exists() else "MISSING"
    size_gb = path.stat().st_size / 1024**3 if path.exists() and path.is_file() else 0
    print(f"{name:32s} {status:8s} {size_gb:8.2f} GB  {path}")

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise RuntimeError("Missing required files: " + ", ".join(missing))

Replogle h5ad                    OK          20.77 GB  /home/lwf/PerturbDiff/data/replogle/perturb_data/finetune_data/nadig_processed_data/replogle.h5ad
selected genes                   OK           0.00 GB  /home/lwf/PerturbDiff/data/replogle/perturb_data/selected_genes/replogle_real_selected_genes.pkl
Replogle perturbation embedding  OK           0.05 GB  /home/lwf/PerturbDiff/data/replogle/perturb_data/gene_names/replogle_gene_emb_dict_perturbation_emb_dict.pkl
